# Question Answering Pipeline

**Authors:** Roberto Herrera and Bach Nguyen  
**Datasets:** HotPotQA and QASPER

We implement a retrieval-augmented extractive question answering pipeline. Stage 1 selects relevant text through chunking, embedding, and ChromaDB retrieval, implementing the assignment’s summarization stage. Stage 2 uses an extractive QA model to predict an answer from the retrieved context.

## 1. Setup

We define dataset locations, load the pipeline dependencies, and create a results directory. Experiment settings and outputs are saved with each run.

In [1]:
import os
import re
import ast
import json
import random
from datetime import datetime
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline
from langchain_huggingface import HuggingFaceEmbeddings

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

HOTPOTQA_PATH = Path(
    "/root/.cache/kagglehub/datasets/"
    "jeromeblanchet/hotpotqa-question-answering-dataset/"
    "versions/1/hotpot_train_v1.1.json"
)
QASPER_DIR = Path(
    "/root/.cache/kagglehub/datasets/"
    "thedevastator/qasper-nlp-questions-and-evidence/versions/2"
)

# change when running on different machine
MACHINE_LABEL = "laptopRH"
RESULTS_DIR = Path("results") / MACHINE_LABEL
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cpu


## 2. Load datasets
HotPotQA provides candidate Wikipedia paragraphs for each question.
QASPER groups questions and answer annotations by research paper.

In [2]:
def parse_cell(s):
    s = re.sub(r",\s*dtype=\w+\)", ")", s)
    s = s.replace("array(", "(")
    return ast.literal_eval(s)


def paper_paragraphs(row):
    full_text = parse_cell(row["full_text"])
    return [
        (section, paragraph)
        for section, paragraphs in zip(
            full_text["section_name"], full_text["paragraphs"]
        )
        for paragraph in paragraphs
        if paragraph.strip()
    ]


with HOTPOTQA_PATH.open(encoding="utf-8") as f:
    hotpot = json.load(f)

qasper = pd.read_csv(QASPER_DIR / "train.csv", dtype={"id": str})

print("HotPotQA questions:", len(hotpot))
print("QASPER papers:", len(qasper))

HotPotQA questions: 90447
QASPER papers: 888


## 3. Chunking

We compare recursive and semantic chunking. Recursive splitting uses a token-counted size cap, while semantic splitting identifies boundaries using a percentile threshold on adjacent text-unit embedding distances. Oversized sentences and semantic groups are split to respect the configured cap. Both methods use the active embedding tokenizer and preserve source metadata and character offsets.

In [3]:
import spacy
from langchain_text_splitters import RecursiveCharacterTextSplitter

nlp = spacy.blank("en")
nlp.add_pipe("sentencizer")


def make_splitter(chunk_size, chunk_overlap, tokenizer, token_limit):
    if not 0 < chunk_size <= token_limit:
        raise ValueError("chunk_size must fit the embedding input limit.")

    special_tokens = tokenizer.num_special_tokens_to_add(pair=False)
    text_budget = chunk_size - special_tokens

    if not 0 <= chunk_overlap < text_budget:
        raise ValueError("Overlap must be smaller than the text token budget.")

    return RecursiveCharacterTextSplitter(
        chunk_size=text_budget,
        chunk_overlap=chunk_overlap,
        length_function=lambda text: len(tokenizer.encode(
            text, add_special_tokens=False, truncation=False
        )),
        add_start_index=True,
    )


def semantic_chunks(text, metadata, embedder, splitter, percentile):
    if not 0 < percentile < 100:
        raise ValueError("percentile must be between 0 and 100")

    units = []
    for sentence in nlp(text).sents:
        if not sentence.text.strip():
            continue
        for piece in splitter.create_documents([sentence.text]):
            start = sentence.start_char + piece.metadata["start_index"]
            units.append((piece.page_content, start))

    if not units:
        return []

    breaks = []
    if len(units) > 1:
        vectors = np.asarray(
            embedder.embed_documents([text for text, _ in units]),
            dtype=float,
        )
        vectors /= np.maximum(
            np.linalg.norm(vectors, axis=1, keepdims=True), 1e-12
        )
        distances = 1 - np.clip(
            np.sum(vectors[:-1] * vectors[1:], axis=1), -1, 1
        )
        threshold = np.percentile(distances, percentile)
        breaks = [
            i + 1 for i, distance in enumerate(distances)
            if distance > threshold
        ]

    docs = []
    first = 0
    for last in breaks + [len(units)]:
        start = units[first][1]
        end = units[last - 1][1] + len(units[last - 1][0])
        pieces = splitter.create_documents(
            [text[start:end]], metadatas=[metadata]
        )
        for piece in pieces:
            piece.metadata["start_index"] += start
            docs.append(piece)
        first = last

    return docs

## 4. Store chunks in ChromaDB

Each chunk is embedded and stored with its source identifier and character range.

HotPotQA uses a separate collection for each question’s candidate paragraphs. QASPER reuses a paper’s collection across its questions. Collections use cosine distance to compare question and chunk embeddings.

In [4]:
from uuid import uuid4
from langchain_chroma import Chroma


def hotpot_paragraphs(example):
    return [
        (" ".join(sentences), {"title": title, "para_id": pid})
        for pid, (title, sentences) in enumerate(example["context"])
    ]


def qasper_paragraphs(row):
    return [
        (paragraph, {"section": section, "para_id": pid})
        for pid, (section, paragraph) in enumerate(paper_paragraphs(row))
    ]


def build_store(paragraphs, embedder, chunking, chunk_size,
                chunk_overlap=0, percentile=None, timings=None):
    def now():
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        return perf_counter()

    started = now()

    if chunking not in {"recursive", "semantic"}:
        raise ValueError("chunking must be 'recursive' or 'semantic'")
    if chunking == "semantic":
        if percentile is None or not 0 < percentile < 100:
            raise ValueError("Semantic chunking requires a percentile in (0, 100).")
        if chunk_overlap != 0:
            raise ValueError("Semantic chunking currently requires zero overlap.")

    tokenizer = embedder._client.tokenizer
    splitter = make_splitter(
        chunk_size, chunk_overlap, tokenizer,
        embedder._client.max_seq_length,
    )

    docs = []
    for text, metadata in paragraphs:
        if chunking == "recursive":
            pieces = splitter.create_documents(
                [text], metadatas=[metadata]
            )
        else:
            pieces = semantic_chunks(
                text, metadata, embedder, splitter, percentile
            )

        for piece in pieces:
            start = piece.metadata["start_index"]
            end = start + len(piece.page_content)
            if start < 0 or text[start:end] != piece.page_content:
                raise ValueError("Chunk offsets do not match the source text.")

            token_count = len(tokenizer.encode(
                piece.page_content, add_special_tokens=True,
                truncation=False,
            ))
            if token_count > chunk_size:
                raise ValueError("A chunk exceeds its configured token cap.")

            piece.metadata.update({
                "end_index": end,
                "chunk_id": f"{metadata['para_id']}:{start}:{end}",
                "embedding_tokens": token_count,
            })
            docs.append(piece)

    chunked = now()
    if timings is not None:
        timings["chunking_s"] = chunked - started
        timings["embedding_store_s"] = 0.0

    store = None
    if docs:
        store = Chroma.from_documents(
            documents=docs,
            embedding=embedder,
            ids=[doc.metadata["chunk_id"] for doc in docs],
            collection_name="retrieval_" + uuid4().hex,
            collection_metadata={"hnsw:space": "cosine"},
        )

    finished = now()
    if timings is not None:
        if docs:
            timings["embedding_store_s"] = finished - chunked
        timings["build_total_s"] = finished - started

    return store

## 5. Retrieve relevant chunks
Single-pass retrieval searches using the original question. Two-hop retrieval uses the first retrieved chunk to expand the question, then searches again for additional chunks. Both return up to k unique chunks. We retain cosine distances and search-stage information to examine whether the second search adds useful evidence. Distances from different searches are not treated as one combined ranking.

In [5]:
def retrieve(store, question, k, method, query_tokenizer,
             max_query_tokens, timings=None):
    def now():
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        return perf_counter()

    started = now()

    if k < 1:
        raise ValueError("k must be at least 1")
    if method not in {"single_pass", "two_hop"}:
        raise ValueError("Unknown retrieval method.")
    if store is None:
        if timings is not None:
            timings["retrieval_s"] = now() - started
        return {"chunks": [], "expanded_query": None}

    def token_count(text):
        return len(query_tokenizer.encode(text, add_special_tokens=True))

    if token_count(question) > max_query_tokens:
        raise ValueError("Question exceeds the embedding input limit.")

    first = store.similarity_search_with_score(
        question, k=k if method == "single_pass" else 1
    )
    selected = [(doc, distance, 1) for doc, distance in first]
    expanded_query = None

    if method == "two_hop" and k > 1 and first:
        evidence = first[0][0].page_content
        offsets = query_tokenizer(
            evidence,
            add_special_tokens=False,
            return_offsets_mapping=True,
            truncation=False,
        )["offset_mapping"]

        end = min(len(offsets), max_query_tokens - token_count(question))
        while end > 0:
            candidate = question + "\n" + evidence[:offsets[end - 1][1]]
            if token_count(candidate) <= max_query_tokens:
                expanded_query = candidate
                break
            end -= 1

        if expanded_query is None:
            raise ValueError("No first-hop evidence fits beside the question.")

        seen = {doc.metadata["chunk_id"] for doc, _, _ in selected}
        second = store.similarity_search_with_score(expanded_query, k=k)

        for doc, distance in second:
            chunk_id = doc.metadata["chunk_id"]
            if chunk_id not in seen:
                selected.append((doc, distance, 2))
                seen.add(chunk_id)
            if len(selected) == k:
                break

    result = {
        "chunks": [
            {
                "rank": rank,
                "text": doc.page_content,
                "metadata": dict(doc.metadata),
                "distance": float(distance),
                "hop": hop,
            }
            for rank, (doc, distance, hop) in enumerate(selected, start=1)
        ],
        "expanded_query": expanded_query,
    }

    if timings is not None:
        timings["retrieval_s"] = now() - started

    return result

## 6. Prepare the QA context

Retrieved chunks are added in retrieval order within the reader’s input budget, including the question and special tokens. If a chunk does not fit, only its fitting prefix is included and packing stops. We record the final context, token count, and retained source ranges to distinguish retrieval failures from evidence lost during packing.

In [6]:
def pack_context(question, chunks, qa_tokenizer,
                 max_tokens=512, timings=None):
    started = perf_counter()

    def input_length(context):
        return len(qa_tokenizer(
            question, context, add_special_tokens=True,
            truncation=False,
        )["input_ids"])

    if input_length("") >= max_tokens:
        raise ValueError("The question leaves no room for QA context.")

    context = ""
    included = []

    for chunk in chunks:
        text = chunk["text"]
        separator = "\n\n" if context else ""
        candidate = context + separator + text
        truncated = False

        if input_length(candidate) > max_tokens:
            offsets = qa_tokenizer(
                text, add_special_tokens=False,
                return_offsets_mapping=True, truncation=False,
            )["offset_mapping"]
            end = min(len(offsets), max_tokens)

            while end > 0:
                prefix = text[:offsets[end - 1][1]]
                candidate = context + separator + prefix
                if input_length(candidate) <= max_tokens:
                    text = prefix
                    break
                end -= 1

            if end == 0:
                break
            truncated = True

        context_start = len(context) + len(separator)
        context = candidate
        metadata = dict(chunk["metadata"])
        metadata["end_index"] = metadata["start_index"] + len(text)

        included.append({
            "rank": chunk["rank"],
            "text": text,
            "metadata": metadata,
            "context_start": context_start,
            "context_end": len(context),
            "truncated": truncated,
        })

        if truncated:
            break

    result = {
        "context": context,
        "input_tokens": input_length(context),
        "included_chunks": included,
    }

    if timings is not None:
        timings["packing_s"] = perf_counter() - started

    return result

## 7. Question answering

We compare three pretrained extractive QA models. Each reader receives the question and packed context and returns an answer span with a confidence score. Model loading and inference are timed separately. Abstention is configurable and recorded for each experiment.

In [60]:
from transformers import pipeline

QA_MODELS = [
    "deepset/tinyroberta-squad2",
    "deepset/roberta-base-squad2",
    "google-bert/bert-large-uncased-whole-word-masking-finetuned-squad",
]

READER_LIMIT = 512
MAX_ANSWER_LEN = 512


def synced_time():
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    return perf_counter()


def load_reader(model_name):
    started = synced_time()
    reader = pipeline(
        "question-answering",
        model=model_name,
        tokenizer=model_name,
        device=0 if DEVICE == "cuda" else -1,
    )
    reader.model.eval()
    return reader, synced_time() - started


def answer_question(reader, question, packed, timings, allow_empty):
    actual_tokens = len(reader.tokenizer(
        question,
        packed["context"],
        add_special_tokens=True,
        truncation=False,
    )["input_ids"])

    if actual_tokens > READER_LIMIT:
        raise ValueError("Packed input exceeds the QA token limit.")
    if actual_tokens != packed["input_tokens"]:
        raise ValueError("Pack the context with this reader's tokenizer.")

    started = synced_time()

    if packed["context"].strip():
        prediction = reader(
            question=question,
            context=packed["context"],
            max_seq_len=READER_LIMIT,
            max_question_len=READER_LIMIT,
            max_answer_len=MAX_ANSWER_LEN,
            doc_stride=0,
            handle_impossible_answer=allow_empty,
        )
    else:
        prediction = {
            "answer": "",
            "score": 0.0,
            "start": 0,
            "end": 0,
        }

    timings["qa_s"] = synced_time() - started

    if prediction["answer"]:
        extracted = packed["context"][
            prediction["start"]:prediction["end"]
        ]
        if extracted != prediction["answer"]:
            raise ValueError("Predicted answer does not match its context span.")

    return prediction

## 8. Answer evaluation

We report normalized exact match and word-token F1 against the best matching reference answer. QASPER annotations are preserved as alternatives; multiple spans within one annotation are concatenated. Answer types are retained for separate reporting because an extractive reader cannot represent every answer type equally well. These are transparent baseline metrics rather than a reproduction of QASPER’s official evaluator.

In [8]:
import string
from collections import Counter


def normalize_answer(text):
    text = text.lower().translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())


def score_answer(prediction, references):
    if not references:
        return {"exact_match": None, "f1": None}

    predicted = normalize_answer(prediction)
    scores = []

    for reference in references:
        gold = normalize_answer(reference)
        p, g = predicted.split(), gold.split()
        overlap = sum((Counter(p) & Counter(g)).values())

        if not p or not g:
            f1 = float(p == g)
        else:
            f1 = 2 * overlap / (len(p) + len(g))

        scores.append((float(predicted == gold), f1))

    return {
        "exact_match": max(em for em, _ in scores),
        "f1": max(f1 for _, f1 in scores),
    }


references = {}

for example in hotpot:
    answer = example["answer"]
    references[("hotpot", str(example["_id"]), str(example["_id"]))] = {
        "answers": [answer],
        "answer_type": "yes_no" if answer.lower() in {"yes", "no"} else "span",
    }

for _, row in qasper.iterrows():
    qas = parse_cell(row["qas"])

    for question_id, annotations in zip(qas["question_id"], qas["answers"]):
        answers, types = [], set()

        for annotation in annotations["answer"]:
            if annotation["unanswerable"]:
                answer, answer_type = "", "unanswerable"
            elif annotation["yes_no"] is not None:
                answer = "yes" if annotation["yes_no"] else "no"
                answer_type = "yes_no"
            elif annotation["extractive_spans"]:
                answer = " ".join(annotation["extractive_spans"])
                answer_type = "extractive"
            elif annotation["free_form_answer"].strip():
                answer = annotation["free_form_answer"]
                answer_type = "free_form"
            else:
                continue

            answers.append(answer)
            types.add(answer_type)

        references[("qasper", str(row["id"]), str(question_id))] = {
            "answers": answers,
            "answer_type": "+".join(sorted(types)) or "missing_reference",
        }


## 9. Evaluation helpers

We report normalized answer exact match and token F1, together with source-paragraph and sentence-level evidence precision, recall and F1 before and after context packing. Sentence coverage requires the complete sentence to be present across retained source ranges. QASPER annotation alternatives are preserved, and evidence that cannot be fully mapped to the available text is reported separately. Results include attempted, scored and failed evaluations, actual input lengths, and stage latency distributions.

In [9]:
import platform
import traceback

K_REPORT = (1, 3, 5, 10)

def clean_text(text):
    return " ".join(text.lower().split())

def make_units(n_hotpot=None, n_papers=None, seed=SEED):
    """None selects all loaded records; integer limits select seeded samples."""
    rng = random.Random(seed)

    def select(items, count):
        if count is None:
            return items
        if not 0 <= count <= len(items):
            raise ValueError("Sample size must be within the dataset size.")
        return rng.sample(items, count)

    units = {"hotpot": [], "qasper": []}

    for example in select(hotpot, n_hotpot):
        spans = {}
        title_to_pid = {}

        for pid, (title, sentences) in enumerate(example["context"]):
            title_to_pid[title] = pid
            position, items = 0, []

            for sentence in sentences:
                left = len(sentence) - len(sentence.lstrip())
                right = len(sentence.rstrip())
                items.append((position + left, position + right))
                position += len(sentence) + 1

            spans[pid] = items

        gold_sentences = set()
        mapped = True

        for title, idx in example["supporting_facts"]:
            pid = title_to_pid.get(title)
            if pid is None or not 0 <= idx < len(spans[pid]):
                mapped = False
            else:
                gold_sentences.add((pid, idx))

        evidence_sets = []
        if mapped and gold_sentences:
            evidence_sets.append({
                "docs": {pid for pid, _ in gold_sentences},
                "sentences": gold_sentences,
            })

        units["hotpot"].append({
            "context_id": str(example["_id"]),
            "paragraphs": hotpot_paragraphs(example),
            "sentence_spans": spans,
            "questions": [{
                "question_id": str(example["_id"]),
                "text": example["question"],
                "evidence_sets": evidence_sets,
                "evidence_status": "ok" if evidence_sets else "unmapped_evidence",
                "unmapped_evidence_items": (
                    0 if mapped else len(example["supporting_facts"])
                ),
            }],
        })

    papers = [row for _, row in qasper.iterrows()]

    for row in select(papers, n_papers):
        paragraphs = paper_paragraphs(row)
        spans, sentence_texts = {}, {}

        for pid, (_, paragraph) in enumerate(paragraphs):
            sentences = [s for s in nlp(paragraph).sents if s.text.strip()]
            spans[pid] = []
            sentence_texts[pid] = []

            for sentence in sentences:
                left = len(sentence.text) - len(sentence.text.lstrip())
                right = len(sentence.text.rstrip())
                spans[pid].append((
                    sentence.start_char + left,
                    sentence.start_char + right,
                ))
                sentence_texts[pid].append(clean_text(sentence.text))

        normalized_paragraphs = [
            clean_text(text) for _, text in paragraphs
        ]
        qas = parse_cell(row["qas"])
        questions = []

        for i, question_id in enumerate(qas["question_id"]):
            alternatives = []
            missing_items = 0
            text_annotations = 0

            for annotation in qas["answers"][i]["answer"]:
                if annotation["unanswerable"]:
                    continue

                evidence = [
                    clean_text(e)
                    for e in annotation["evidence"]
                    if e.strip()
                ]
                if not evidence:
                    continue

                text_annotations += 1
                gold_docs, gold_sentences = set(), set()
                complete = True

                for evidence_text in evidence:
                    if evidence_text.startswith("float selected"):
                        missing_items += 1
                        complete = False
                        continue

                    matches = {
                        pid for pid, paragraph in enumerate(normalized_paragraphs)
                        if evidence_text in paragraph
                    }
                    matched_sentences = {
                        (pid, idx)
                        for pid in matches
                        for idx, sentence in enumerate(sentence_texts[pid])
                        if sentence and sentence in evidence_text
                    }

                    if not matches or not matched_sentences:
                        missing_items += 1
                        complete = False
                        continue

                    gold_docs.update(matches)
                    gold_sentences.update(matched_sentences)

                # Never score an incomplete annotation as complete evidence.
                if complete and gold_sentences:
                    alternatives.append({
                        "docs": gold_docs,
                        "sentences": gold_sentences,
                    })

            if alternatives:
                status = "ok" if missing_items == 0 else "partially_mapped_alternatives"
            elif text_annotations:
                status = "unmapped_evidence"
            else:
                status = "no_answerable_text_evidence"

            questions.append({
                "question_id": str(question_id),
                "text": qas["question"][i],
                "evidence_sets": alternatives,
                "evidence_status": status,
                "unmapped_evidence_items": missing_items,
            })

        units["qasper"].append({
            "context_id": str(row["id"]),
            "paragraphs": qasper_paragraphs(row),
            "sentence_spans": spans,
            "questions": questions,
        })

    return units

def beginning_chunks(unit):
    return {
        "chunks": [
            {
                "rank": rank,
                "text": text,
                "metadata": {
                    **metadata,
                    "start_index": 0,
                    "end_index": len(text),
                    "chunk_id": f"{metadata['para_id']}:0:{len(text)}",
                },
                "distance": None,
                "hop": 0,
            }
            for rank, (text, metadata) in enumerate(unit["paragraphs"], start=1)
        ],
        "expanded_query": None,
    }


def prf(gold, found):
    if not gold:
        return None, None, None
    hits = len(gold & found)
    precision = hits / len(found) if found else 0.0
    recall = hits / len(gold)
    f1 = 2 * hits / (len(gold) + len(found))
    return precision, recall, f1


def covered_sentences(chunks, spans):
    ranges = {}
    for chunk in chunks:
        metadata = chunk["metadata"]
        ranges.setdefault(metadata["para_id"], []).append(
            (metadata["start_index"], metadata["end_index"])
        )

    covered = set()
    for pid, intervals in ranges.items():
        merged = []
        for start, end in sorted(intervals):
            if merged and start <= merged[-1][1]:
                merged[-1] = (merged[-1][0], max(end, merged[-1][1]))
            else:
                merged.append((start, end))

        for idx, (start, end) in enumerate(spans[pid]):
            if end > start and any(a <= start and end <= b for a, b in merged):
                covered.add((pid, idx))

    return covered


EVIDENCE_COLUMNS = [
    f"{stage}_{level}_{metric}"
    for stage in ("retrieved", "window")
    for level in ("doc", "sent")
    for metric in ("precision", "recall", "f1")
]


def evidence_metrics(unit, question, chunks, stage):
    out = {name: None for name in EVIDENCE_COLUMNS if name.startswith(stage)}
    alternatives = question["evidence_sets"]
    if not alternatives:
        return out

    found_docs = {c["metadata"]["para_id"] for c in chunks}
    found_sentences = covered_sentences(chunks, unit["sentence_spans"])

    for level, found in (("doc", found_docs), ("sent", found_sentences)):
        key = "docs" if level == "doc" else "sentences"
        scores = [prf(alt[key], found) for alt in alternatives]
        # Keep precision and recall from the same best-F1 annotation.
        p, r, f = max(scores, key=lambda values: values[2])
        out.update({
            f"{stage}_{level}_precision": p,
            f"{stage}_{level}_recall": r,
            f"{stage}_{level}_f1": f,
        })

    return out

def machine_info():
    return {
        "label": MACHINE_LABEL,
        "platform": platform.platform(),
        "python": platform.python_version(),
        "logical_processors": os.cpu_count(),
        "device": DEVICE,
        "torch": torch.__version__,
        "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None,
    }


KEYS = ["dataset", "embedder", "chunking", "cap", "search", "k", "qa"]


def summarize(df):
    if df.empty:
        return pd.DataFrame()

    data = df.copy()
    data["failed"] = data["status"].ne("ok")
    data["succeeded"] = ~data["failed"]

    for column in EVIDENCE_COLUMNS:
        if column not in data:
            data[column] = np.nan

    timings = [
        "build_total_s", "build_amortized_s", "retrieval_s",
        "packing_s", "qa_s", "reader_stage_s", "pipeline_stage_sum_s",
    ]
    for column in timings + ["input_tokens", "n_chunks"]:
        if column not in data:
            data[column] = np.nan

    aggregations = {
        "questions": ("status", "size"),
        "successful_questions": ("succeeded", "sum"),
        "errors": ("failed", "sum"),
        "scored_questions": ("exact_match", "count"),
        "exact_match": ("exact_match", "mean"),
        "f1": ("f1", "mean"),
        "confidence_passes": ("confidence_pass", "sum"),
        "mean_input_tokens": ("input_tokens", "mean"),
        "mean_packed_chunks": ("n_chunks", "mean"),
    }

    for column in EVIDENCE_COLUMNS:
        aggregations[column] = (column, "mean")
        aggregations[column + "_n"] = (column, "count")

    for column in timings:
        aggregations[column + "_mean"] = (column, "mean")

    aggregations["pipeline_stage_sum_s_p95"] = (
        "pipeline_stage_sum_s", lambda values: values.quantile(0.95)
    )

    return data.groupby(KEYS, dropna=False).agg(
        **aggregations
    ).reset_index()

## 10. Experiment runner
Each context collection is built once per embedding and chunking configuration, and retrieval outputs are saved for reuse across readers. QA models are loaded one at a time and evaluated at each requested retrieval depth. Every record includes the question, reference answers, retrieved chunks, retained context, evidence metrics, prediction and stage timings. Construction cost is reported separately. Retrieval plus reader-stage time is a sum of separately measured stages, not a directly measured live request latency. Completed retrieval blocks and QA records can be reused when resuming the same run directory.

In [10]:
import gc

CONFIDENCE_THRESHOLD = 0.5


def run_experiment(name, units, embedders, chunkings, caps, searches, qa_models,
                   ks=K_REPORT, percentile=90, run_dir=None):
    ks = sorted(set(ks))
    if not ks or any(k < 1 for k in ks):
        raise ValueError("Provide positive retrieval depths.")
    if not set(searches) <= {"single_pass", "two_hop", "beginning"}:
        raise ValueError("Unknown search method.")
    if not qa_models:
        raise ValueError("Provide at least one QA model.")

    if run_dir is None:
        run_dir = RESULTS_DIR / f"{name}_{datetime.now():%Y%m%d_%H%M%S_%f}"
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    cache_dir = run_dir / "retrieval"
    cache_dir.mkdir(exist_ok=True)

    config = {
        "name": name,
        "embedders": list(embedders),
        "chunkings": list(chunkings),
        "caps": list(caps),
        "searches": list(searches),
        "qa_models": list(qa_models),
        "ks": ks,
        "percentile": percentile,
        "chunk_overlap": 0,
        "reader_limit": READER_LIMIT,
        "max_answer_len": MAX_ANSWER_LEN,
        "confidence_threshold": CONFIDENCE_THRESHOLD,
        "seed": SEED,
        "timing_mode": "saved_retrieval_plus_measured_reader_stage",
        "sample_ids": {
            dataset: [
                [unit["context_id"],
                 [q["question_id"] for q in unit["questions"]]]
                for unit in dataset_units
            ]
            for dataset, dataset_units in units.items()
        },
    }

    config_path = run_dir / "config.json"
    if config_path.exists():
        if json.loads(config_path.read_text()) != config:
            raise ValueError("Resume settings differ. Use a new run directory.")
    else:
        config_path.write_text(json.dumps(config, indent=2))
        (run_dir / "machine.json").write_text(
            json.dumps(machine_info(), indent=2)
        )

    lookup = {
        (dataset, unit["context_id"], question["question_id"]): (unit, question)
        for dataset, dataset_units in units.items()
        for unit in dataset_units
        for question in unit["questions"]
    }

    def cache_record(dataset, unit, question, embedder_name,
                     chunking, cap, search, k, build_times,
                     retrieved=None, retrieval_s=None, error=None):
        refs = references.get(
            (dataset, unit["context_id"], question["question_id"]),
            {"answers": [], "answer_type": "missing_reference"},
        )
        record = {
            "dataset": dataset,
            "context_id": unit["context_id"],
            "question_id": question["question_id"],
            "question": question["text"],
            "embedder": embedder_name,
            "chunking": chunking,
            "cap": cap,
            "search": search,
            "k": k,
            "answer_type": refs["answer_type"],
            "reference_answers": refs["answers"],
            "evidence_status": question["evidence_status"],
            "unmapped_evidence_items": question["unmapped_evidence_items"],
            "empty_context": not bool(unit["paragraphs"]),
            "build_total_s": build_times.get("build_total_s"),
            "chunking_s": build_times.get("chunking_s"),
            "embedding_store_s": build_times.get("embedding_store_s"),
            "build_amortized_s": (
                build_times["build_total_s"] / len(unit["questions"])
                if "build_total_s" in build_times else None
            ),
            "retrieval_s": retrieval_s,
            "retrieved": retrieved,
            "retrieval_error": error,
        }
        if retrieved is not None:
            record.update(evidence_metrics(
                unit, question, retrieved["chunks"], "retrieved"
            ))
        return record

    cache_paths = []
    retrieval_methods = [s for s in searches if s != "beginning"]

    # Run the beginning-of-context baseline once, independently of embedders.
    if "beginning" in searches:
        path = cache_dir / "beginning.jsonl"
        cache_paths.append(path)

        if not path.exists():
            temporary = path.with_suffix(".partial")
            with temporary.open("w", encoding="utf-8") as output:
                for dataset, dataset_units in units.items():
                    for unit in dataset_units:
                        for question in unit["questions"]:
                            record = cache_record(
                                dataset, unit, question, "none", "none", None,
                                "beginning", None,
                                {"build_total_s": 0.0, "chunking_s": 0.0,
                                 "embedding_store_s": 0.0},
                                beginning_chunks(unit), 0.0,
                            )
                            output.write(json.dumps(record, ensure_ascii=False) + "\n")
            temporary.replace(path)

    model_log = run_dir / "model_loads.jsonl"
    started_run = perf_counter()

    for embedder_name in embedders if retrieval_methods else []:
        model_id = (
            embedder_name if "/" in embedder_name
            else "sentence-transformers/" + embedder_name
        )
        started = synced_time()
        embedder = HuggingFaceEmbeddings(
            model_name=model_id,
            model_kwargs={"device": DEVICE},
        )
        load_s = synced_time() - started
        embedder.embed_query("Warm-up query.")
        token_limit = embedder._client.max_seq_length
        tokenizer = embedder._client.tokenizer

        with model_log.open("a", encoding="utf-8") as output:
            output.write(json.dumps({
                "stage": "embedding", "model": model_id,
                "load_s": load_s, "token_limit": token_limit,
            }) + "\n")

        for chunking in chunkings:
            for cap in caps:
                if cap > token_limit:
                    print(f"Skipped: {model_id}, cap {cap} > {token_limit}")
                    continue

                block_id = (
                    f"{model_id.replace('/', '__')}_{chunking}_{cap}"
                )
                path = cache_dir / f"{block_id}.jsonl"
                cache_paths.append(path)

                if path.exists():
                    print("Reusing:", path.name, flush=True)
                    continue

                temporary = path.with_suffix(".partial")
                count = 0

                with temporary.open("w", encoding="utf-8") as output:
                    for dataset, dataset_units in units.items():
                        for unit in dataset_units:
                            if not unit["questions"]:
                                continue

                            store, build_error, build_times = None, None, {}
                            try:
                                try:
                                    store = build_store(
                                        unit["paragraphs"], embedder, chunking,
                                        cap, 0,
                                        percentile if chunking == "semantic" else None,
                                        build_times,
                                    )
                                except Exception:
                                    build_error = traceback.format_exc()

                                for question in unit["questions"]:
                                    for search in retrieval_methods:
                                        for k in ks:
                                            retrieved, error, times = None, build_error, {}
                                            if error is None:
                                                try:
                                                    retrieved = retrieve(
                                                        store, question["text"], k,
                                                        search, tokenizer, token_limit,
                                                        times,
                                                    )
                                                except Exception:
                                                    error = traceback.format_exc()

                                            record = cache_record(
                                                dataset, unit, question, model_id,
                                                chunking, cap, search, k,
                                                build_times, retrieved,
                                                times.get("retrieval_s"), error,
                                            )
                                            output.write(
                                                json.dumps(record, ensure_ascii=False) + "\n"
                                            )
                                            count += 1

                                    output.flush()
                                    if count % 100 == 0:
                                        print(block_id, count, "retrieval records",
                                              flush=True)
                            finally:
                                if store is not None:
                                    store.delete_collection()

                temporary.replace(path)
                print("Saved:", path.name, flush=True)

        del tokenizer, embedder
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    def record_key(record):
        return (
            record["dataset"], record["context_id"], record["question_id"],
            record["embedder"], record["chunking"], record["cap"],
            record["search"], record["k"], record["qa"],
        )

    predictions_path = run_dir / "records.jsonl"
    completed = set()

    # Remove only an incomplete trailing record left by an interruption.
    if predictions_path.exists():
        with predictions_path.open("rb+") as source:
            while True:
                position = source.tell()
                line = source.readline()
                if not line:
                    break
                try:
                    saved = json.loads(line)
                except json.JSONDecodeError:
                    if source.read():
                        raise ValueError("Corrupt record inside records.jsonl.")
                    source.seek(position)
                    source.truncate()
                    break
                completed.add(record_key(saved))

    # Release readers cached by the previous implementation, if present.
    if "READERS" in globals():
        READERS.clear()
    gc.collect()

    with predictions_path.open("a", encoding="utf-8") as output:
        for qa_name in qa_models:
            reader, load_s = load_reader(qa_name)
            reader(question="What is it?", context="It is a test.",
                   max_seq_len=READER_LIMIT)

            with model_log.open("a", encoding="utf-8") as log:
                log.write(json.dumps({
                    "stage": "qa", "model": qa_name, "load_s": load_s,
                }) + "\n")

            for path in cache_paths:
                with path.open(encoding="utf-8") as source:
                    for line in source:
                        saved = json.loads(line)
                        record = {**saved, "qa": qa_name}
                        key = record_key(record)
                        if key in completed:
                            continue

                        unit, question = lookup[(
                            record["dataset"], record["context_id"],
                            record["question_id"],
                        )]
                        allow_empty = (
                            record["dataset"] == "qasper"
                            and "squad2" in qa_name
                        )
                        record.update({
                            "status": "ok",
                            "allow_empty": allow_empty,
                            "confidence_pass": False,
                        })

                        try:
                            if record["retrieval_error"]:
                                raise RuntimeError(record["retrieval_error"])

                            times = {}
                            started = synced_time()
                            packed = pack_context(
                                question["text"], record["retrieved"]["chunks"],
                                reader.tokenizer, READER_LIMIT, times,
                            )
                            prediction = answer_question(
                                reader, question["text"], packed, times,
                                allow_empty,
                            )
                            reader_stage_s = synced_time() - started

                            record.update({
                                "prediction": prediction,
                                "packed": packed,
                                "confidence": float(prediction["score"]),
                                "confidence_pass": (
                                    bool(packed["context"].strip())
                                    and prediction["score"] >= CONFIDENCE_THRESHOLD
                                ),
                                "packing_s": times["packing_s"],
                                "qa_s": times["qa_s"],
                                "reader_stage_s": reader_stage_s,
                                "pipeline_stage_sum_s": (
                                    record["retrieval_s"] + reader_stage_s
                                ),
                                "n_chunks": len(packed["included_chunks"]),
                                "input_tokens": packed["input_tokens"],
                                "truncated_last": any(
                                    c["truncated"] for c in packed["included_chunks"]
                                ),
                                **evidence_metrics(
                                    unit, question, packed["included_chunks"], "window"
                                ),
                                **score_answer(
                                    prediction["answer"], record["reference_answers"]
                                ),
                            })
                        except Exception:
                            has_refs = bool(record["reference_answers"])
                            record.update({
                                "status": "error",
                                "error": traceback.format_exc(),
                                "exact_match": 0.0 if has_refs else None,
                                "f1": 0.0 if has_refs else None,
                            })

                        output.write(json.dumps(record, ensure_ascii=False) + "\n")
                        output.flush()
                        completed.add(key)

                        if len(completed) % 100 == 0:
                            print(len(completed), "QA records saved", flush=True)

            del reader
            gc.collect()
            if DEVICE == "cuda":
                torch.cuda.empty_cache()

    excluded_fields = {
        "retrieved", "packed", "prediction", "reference_answers",
        "question", "error", "retrieval_error",
    }
    rows = []
    with predictions_path.open(encoding="utf-8") as source:
        for line in source:
            record = json.loads(line)
            rows.append({
                key: value for key, value in record.items()
                if key not in excluded_fields
            })

    df = pd.DataFrame(rows)
    df.to_csv(run_dir / "question_metrics.csv", index=False)
    summary = summarize(df)
    summary.to_csv(run_dir / "summary.csv", index=False)

    print("Finished in", round((perf_counter() - started_run) / 60, 1), "minutes")
    print("Saved:", run_dir)
    return df, summary

## 11. Experiments
All experiments run from one plan. Each run is saved in its own folder, and a run whose folder already holds finished results for the same settings and the same questions is reused. The combined table covers every configuration. Set QUICK to True to reproduce the pipeline on a small sample in minutes.

In [14]:
import json
from time import perf_counter

QUICK = False

L6, L12, MPNET = "all-MiniLM-L6-v2", "all-MiniLM-L12-v2", "all-mpnet-base-v2"
TINY, ROBERTA, BERT = QA_MODELS
BOTH = ["single_pass", "two_hop"]
ALL_K = (1, 3, 5, 10)
PERCENTILE = 90

PLAN = [
    dict(name="A0_baseline", dir="design_experiments_v4/A0_baseline", embedders=[L6],
         chunkings=["recursive"], caps=[128], searches=["beginning"], ks=(3, 10), qa=[TINY]),
    dict(name="A1_recursive", dir="design_experiments_v4/A1_recursive", embedders=[L6],
         chunkings=["recursive"], caps=[128], searches=BOTH, ks=(3, 10), qa=[TINY]),
    dict(name="A2_semantic", dir="design_experiments_v4/A2_semantic", embedders=[L6],
         chunkings=["semantic"], caps=[128], searches=BOTH, ks=(3, 10), qa=[TINY]),
    dict(name="P1_budget_k1_k5", dir="first_pass_v1/P1_budget_k1_k5", embedders=[L6],
         chunkings=["recursive"], caps=[128], searches=BOTH, ks=(1, 5), qa=[TINY]),
    dict(name="C2_chunk_size", dir="complete_v1/C2_chunk_size", embedders=[L6],
         chunkings=["recursive"], caps=[64, 256], searches=BOTH, ks=ALL_K, qa=[TINY]),
    dict(name="C3_semantic", dir="complete_v1/C3_semantic", embedders=[L6],
         chunkings=["semantic"], caps=[128], searches=BOTH, ks=(1, 5), qa=[TINY]),
    dict(name="C4_embedders", dir="complete_v1/C4_embedders", embedders=[L12, MPNET],
         chunkings=["recursive"], caps=[128], searches=BOTH, ks=ALL_K, qa=[TINY]),
    dict(name="C5_readers", dir="complete_v1/C5_readers", embedders=[L6],
         chunkings=["recursive"], caps=[128], searches=["single_pass"], ks=ALL_K,
         qa=[ROBERTA, BERT]),
]

units = make_units(n_hotpot=10 if QUICK else 100, n_papers=3 if QUICK else 20, seed=SEED)
FINAL_DIR = RESULTS_DIR / ("final_quick" if QUICK else "final_results")
FINAL_DIR.mkdir(parents=True, exist_ok=True)

n_questions = sum(len(u["questions"]) for dataset_units in units.values() for u in dataset_units)
sample_ids = {
    dataset: [[u["context_id"], [q["question_id"] for q in u["questions"]]] for u in dataset_units]
    for dataset, dataset_units in units.items()
}


def expected_evaluations(run):
    retrieval = sum(s != "beginning" for s in run["searches"])
    configs = (
        (len(run["qa"]) if "beginning" in run["searches"] else 0)
        + len(run["embedders"]) * len(run["chunkings"]) * len(run["caps"])
        * retrieval * len(run["ks"]) * len(run["qa"])
    )
    return n_questions * configs


def same_settings(run_dir, run):
    path = run_dir / "config.json"
    if not path.exists():
        return None
    saved = json.loads(path.read_text())
    wanted = {
        "embedders": run["embedders"], "chunkings": run["chunkings"],
        "caps": list(run["caps"]), "searches": run["searches"],
        "qa_models": run["qa"], "ks": sorted(set(run["ks"])),
        "percentile": PERCENTILE, "sample_ids": sample_ids,
    }
    return all(saved.get(key) == value for key, value in wanted.items())


started_all = perf_counter()
summaries, status = [], []

for run in PLAN:
    run_dir = RESULTS_DIR / (f"final_quick/{run['name']}" if QUICK else run["dir"])
    expected = expected_evaluations(run)
    match = same_settings(run_dir, run)

    if match is False:
        raise ValueError(
            f"{run['name']}: {run_dir} holds different settings or questions. "
            "Move that folder away or change the plan."
        )

    summary = None
    if match and (run_dir / "summary.csv").exists():
        saved = pd.read_csv(run_dir / "summary.csv")
        if int(saved["questions"].sum()) == expected:
            summary = saved
            status.append((run["name"], "reused", expected))
            print("Reused (already computed):", run["name"], flush=True)

    if summary is None:
        print("Computing:", run["name"], "| expected evaluations", expected, flush=True)
        _, summary = run_experiment(
            name=run["name"], units=units, embedders=run["embedders"],
            chunkings=run["chunkings"], caps=run["caps"], searches=run["searches"],
            qa_models=run["qa"], ks=run["ks"], percentile=PERCENTILE, run_dir=run_dir,
        )
        assert int(summary["questions"].sum()) == expected, run["name"]
        status.append((run["name"], "computed", expected))

    summaries.append(summary)

combined = pd.concat(summaries, ignore_index=True)
combined.to_csv(FINAL_DIR / "combined_summary.csv", index=False)
(FINAL_DIR / "plan.json").write_text(json.dumps(
    {"quick": QUICK, "questions": n_questions, "runs": PLAN}, indent=2, default=list
))

print("\nRuns:", status)
print("Total evaluations:", int(combined["questions"].sum()),
      "| errors:", int(combined["errors"].sum()),
      "| minutes this session:", round((perf_counter() - started_all) / 60, 1))

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)
view = combined.copy()
view["embedder"] = view["embedder"].astype(str).str.split("/").str[-1]
view["qa"] = view["qa"].astype(str).str.split("/").str[-1].str[:22]
display(view.sort_values(
    ["dataset", "embedder", "chunking", "cap", "search", "qa", "k"]
)[[
    "dataset", "embedder", "chunking", "cap", "search", "k", "qa",
    "questions", "errors", "exact_match", "f1",
    "retrieved_doc_recall", "window_doc_recall", "window_doc_precision",
    "window_sent_recall", "mean_input_tokens", "mean_packed_chunks",
    "build_amortized_s_mean", "retrieval_s_mean", "packing_s_mean",
    "qa_s_mean", "pipeline_stage_sum_s_mean",
]].round(3))

Reused (already computed): A0_baseline
Reused (already computed): A1_recursive
Reused (already computed): A2_semantic
Reused (already computed): P1_budget_k1_k5
Computing: C2_chunk_size | expected evaluations 2512
sentence-transformers__all-MiniLM-L6-v2_recursive_64 200 retrieval records
sentence-transformers__all-MiniLM-L6-v2_recursive_64 400 retrieval records
sentence-transformers__all-MiniLM-L6-v2_recursive_64 600 retrieval records
sentence-transformers__all-MiniLM-L6-v2_recursive_64 800 retrieval records
sentence-transformers__all-MiniLM-L6-v2_recursive_64 1000 retrieval records
sentence-transformers__all-MiniLM-L6-v2_recursive_64 1200 retrieval records
Saved: sentence-transformers__all-MiniLM-L6-v2_recursive_64.jsonl
sentence-transformers__all-MiniLM-L6-v2_recursive_256 200 retrieval records
sentence-transformers__all-MiniLM-L6-v2_recursive_256 400 retrieval records
sentence-transformers__all-MiniLM-L6-v2_recursive_256 600 retrieval records
sentence-transformers__all-MiniLM-L6-v2_

Device set to use cpu
Token indices sequence length is longer than the specified maximum sequence length for this model (555 > 512). Running this sequence through the model will result in indexing errors


100 QA records saved
200 QA records saved
300 QA records saved
400 QA records saved
500 QA records saved
600 QA records saved
700 QA records saved
800 QA records saved
900 QA records saved
1000 QA records saved
1100 QA records saved
1200 QA records saved
1300 QA records saved
1400 QA records saved
1500 QA records saved
1600 QA records saved
1700 QA records saved
1800 QA records saved
1900 QA records saved
2000 QA records saved
2100 QA records saved
2200 QA records saved
2300 QA records saved
2400 QA records saved
2500 QA records saved
Finished in 22.1 minutes
Saved: results/laptopRH/complete_v1/C2_chunk_size
Computing: C3_semantic | expected evaluations 628
sentence-transformers__all-MiniLM-L6-v2_semantic_128 100 retrieval records
sentence-transformers__all-MiniLM-L6-v2_semantic_128 200 retrieval records
sentence-transformers__all-MiniLM-L6-v2_semantic_128 300 retrieval records
sentence-transformers__all-MiniLM-L6-v2_semantic_128 400 retrieval records
sentence-transformers__all-MiniLM-

Device set to use cpu
Token indices sequence length is longer than the specified maximum sequence length for this model (516 > 512). Running this sequence through the model will result in indexing errors


100 QA records saved
200 QA records saved
300 QA records saved
400 QA records saved
500 QA records saved
600 QA records saved
Finished in 9.3 minutes
Saved: results/laptopRH/complete_v1/C3_semantic
Computing: C4_embedders | expected evaluations 2512
sentence-transformers__all-MiniLM-L12-v2_recursive_128 200 retrieval records
sentence-transformers__all-MiniLM-L12-v2_recursive_128 400 retrieval records
sentence-transformers__all-MiniLM-L12-v2_recursive_128 600 retrieval records
sentence-transformers__all-MiniLM-L12-v2_recursive_128 800 retrieval records
sentence-transformers__all-MiniLM-L12-v2_recursive_128 1000 retrieval records
sentence-transformers__all-MiniLM-L12-v2_recursive_128 1200 retrieval records
Saved: sentence-transformers__all-MiniLM-L12-v2_recursive_128.jsonl
sentence-transformers__all-mpnet-base-v2_recursive_128 200 retrieval records
sentence-transformers__all-mpnet-base-v2_recursive_128 400 retrieval records
sentence-transformers__all-mpnet-base-v2_recursive_128 600 retri

...

Device set to use cpu
Token indices sequence length is longer than the specified maximum sequence length for this model (552 > 512). Running this sequence through the model will result in indexing errors


100 QA records saved
200 QA records saved
300 QA records saved
400 QA records saved
500 QA records saved
600 QA records saved


Some weights of the model checkpoint at google-bert/bert-large-uncased-whole-word-masking-finetuned-squad were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu
Token indices sequence length is longer than the specified maximum sequence length for this model (522 > 512). Running this sequence through the model will result in indexing errors


700 QA records saved
800 QA records saved
900 QA records saved
1000 QA records saved
1100 QA records saved
1200 QA records saved
Finished in 46.5 minutes
Saved: results/laptopRH/complete_v1/C5_readers

Runs: [('A0_baseline', 'reused', 157), ('A1_recursive', 'reused', 628), ('A2_semantic', 'reused', 628), ('P1_budget_k1_k5', 'reused', 628), ('C2_chunk_size', 'computed', 2512), ('C3_semantic', 'computed', 628), ('C4_embedders', 'computed', 2512), ('C5_readers', 'computed', 1256)]
Total evaluations: 8949 | errors: 0 | minutes this session: 111.6


,dataset,embedder,chunking,cap,search,k,qa,questions,errors,exact_match,f1,retrieved_doc_recall,window_doc_recall,window_doc_precision,window_sent_recall,mean_input_tokens,mean_packed_chunks,build_amortized_s_mean,retrieval_s_mean,packing_s_mean,qa_s_mean,pipeline_stage_sum_s_mean
66,hotpot,all-MiniLM-L12-v2,recursive,128.0,single_pass,1.0,tinyroberta-squad2,100,0,0.220,0.287,0.470,0.470,0.940,0.442,114.140,1.000,1.268,0.083,0.002,0.145,0.232
67,hotpot,all-MiniLM-L12-v2,recursive,128.0,single_pass,3.0,tinyroberta-squad2,100,0,0.310,0.348,0.725,0.725,0.510,0.688,284.630,3.000,1.268,0.074,0.006,0.300,0.381
68,hotpot,all-MiniLM-L12-v2,recursive,128.0,single_pass,5.0,tinyroberta-squad2,100,0,0.290,0.349,0.810,0.800,0.349,0.758,447.260,4.930,1.268,0.064,0.047,0.436,0.549
69,hotpot,all-MiniLM-L12-v2,recursive,128.0,single_pass,10.0,tinyroberta-squad2,100,0,0.290,0.335,0.960,0.845,0.315,0.789,511.950,5.990,1.268,0.071,0.136,0.453,0.661
70,hotpot,all-MiniLM-L12-v2,recursive,128.0,two_hop,1.0,tinyroberta-squad2,100,0,0.220,0.287,0.470,0.470,0.940,0.442,114.140,1.000,1.268,0.069,0.002,0.149,0.220
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,qasper,all-mpnet-base-v2,recursive,128.0,two_hop,1.0,tinyroberta-squad2,57,0,0.070,0.113,0.146,0.146,0.194,0.102,70.193,0.947,4.057,0.072,0.001,0.069,0.143
95,qasper,all-mpnet-base-v2,recursive,128.0,two_hop,3.0,tinyroberta-squad2,57,0,0.123,0.167,0.236,0.236,0.111,0.186,226.772,2.842,4.057,0.209,0.005,0.185,0.400
96,qasper,all-mpnet-base-v2,recursive,128.0,two_hop,5.0,tinyroberta-squad2,57,0,0.123,0.151,0.306,0.306,0.093,0.270,390.404,4.737,4.057,0.227,0.024,0.298,0.551
97,qasper,all-mpnet-base-v2,recursive,128.0,two_hop,10.0,tinyroberta-squad2,57,0,0.105,0.134,0.542,0.403,0.098,0.325,485.632,6.298,4.057,0.229,0.116,0.357,0.704


In [ ]:
combined_units = make_units(n_hotpot=100, n_papers=20, seed=SEED)

_, combined_summary = run_experiment(
    name="combined_best",
    units=combined_units,
    embedders=["sentence-transformers/all-MiniLM-L6-v2"],
    chunkings=["recursive"],
    caps=[256],
    searches=["two_hop"],
    qa_models=[
        "deepset/roberta-base-squad2",
        "google-bert/bert-large-uncased-whole-word-masking-finetuned-squad",
    ],
    ks=[3, 5],
    percentile=90,
    run_dir=RESULTS_DIR / "combined_best",
)

display(combined_summary[[
    "dataset", "cap", "search", "k", "qa", "questions", "errors", "exact_match", "f1",
    "confidence_passes", "window_doc_recall", "window_doc_f1", "qa_s_mean", "pipeline_stage_sum_s_mean",
]].round(3))

sentence-transformers__all-MiniLM-L6-v2_recursive_256 100 retrieval records


In [ ]:
import gc
import json
import time
import threading
import psutil
import pandas as pd
from pathlib import Path

READERS = [
    "deepset/tinyroberta-squad2",
    "deepset/roberta-base-squad2",
    "google-bert/bert-large-uncased-whole-word-masking-finetuned-squad",
]

test_units = make_units(
    n_hotpot=100,
    n_papers=0,import gc
import time
import threading
import psutil
import pandas as pd

READERS = [
    "deepset/tinyroberta-squad2",
    "deepset/roberta-base-squad2",
    "google-bert/bert-large-uncased-whole-word-masking-finetuned-squad",
]

test_units = make_units(
    n_hotpot=100,
    n_papers=0,
    seed=SEED,
)

process = psutil.Process()
quality_results = []
memory_results = []

def run_profiled(reader):
    reader_name = reader.split("/")[-1]
    run_dir = RESULTS_DIR / f"original_reader_{reader_name}"

    samples = []
    stop = threading.Event()

    def monitor_memory():
        while not stop.is_set():
            samples.append(
                process.memory_info().rss / (1024 ** 2)
            )
            time.sleep(0.25)

    gc.collect()

    ram_before = process.memory_info().rss / (1024 ** 2)
    monitor = threading.Thread(
        target=monitor_memory,
        daemon=True,
    )
    monitor.start()

    start = time.perf_counter()

    try:
        _, summary = run_experiment(
            name=f"original_{reader_name}",
            units=test_units,
            embedders=["sentence-transformers/all-MiniLM-L6-v2"],
            chunkings=["recursive"],
            caps=[128],
            searches=["single_pass"],
            qa_models=[reader],
            ks=[1, 3, 5, 10],
            percentile=90,
            run_dir=run_dir,
        )
    finally:
        stop.set()
        monitor.join(timeout=2)

    elapsed = time.perf_counter() - start
    ram_after = process.memory_info().rss / (1024 ** 2)
    peak_ram = max(samples) if samples else ram_before

    summary = summary.copy()
    summary["reader_model"] = reader_name
    quality_results.append(summary)

    memory_results.append({
        "reader_model": reader_name,
        "wall_time_s": elapsed,
        "ram_before_mb": ram_before,
        "peak_ram_mb": peak_ram,
        "ram_after_mb": ram_after,
        "additional_peak_ram_mb": peak_ram - ram_before,
    })

    del summary
    gc.collect()


for reader in READERS:
    run_profiled(reader)

quality = pd.concat(quality_results, ignore_index=True)
memory = pd.DataFrame(memory_results)

quality.to_csv(
    RESULTS_DIR / "original_reader_quality.csv",
    index=False,
)

memory.to_csv(
    RESULTS_DIR / "original_reader_memory.csv",
    index=False,
)

display(
    quality[
        [
            "dataset", "reader_model", "k",
            "f1", "exact_match",
            "mean_input_tokens",
            "qa_s_mean",
            "pipeline_stage_sum_s_mean",
        ]
    ].sort_values(["reader_model", "k"])
)

display(memory)
    seed=SEED,
)

memory_results = []
quality_results = []

process = psutil.Process()

def run_with_peak_memory(label, **kwargs):
    samples = []
    stop = threading.Event()

    def monitor():
        while not stop.is_set():
            samples.append(process.memory_info().rss / (1024 ** 2))
            time.sleep(0.25)

    before = process.memory_info().rss / (1024 ** 2)
    thread = threading.Thread(target=monitor, daemon=True)
    thread.start()

    start = time.perf_counter()

    try:
        _, result = run_experiment(**kwargs)
    finally:
        stop.set()
        thread.join(timeout=2)

    elapsed = time.perf_counter() - start
    after = process.memory_info().rss / (1024 ** 2)

    memory_results.append({
        "reader": label,
        "wall_time_s": elapsed,
        "ram_before_mb": before,
        "peak_ram_mb": max(samples) if samples else before,
        "ram_after_mb": after,
        "additional_peak_ram_mb": (
            max(samples) - before if samples else 0
        ),
    })

    return result


for reader in READERS:
    reader_name = reader.split("/")[-1]
    run_dir = RESULTS_DIR / f"focused_all_readers_{reader_name}"

    result = run_with_peak_memory(
        label=reader_name,
        name=f"focused_{reader_name}",
        units=test_units,
        embedders=["sentence-transformers/all-MiniLM-L6-v2"],
        chunkings=["recursive"],
        caps=[128],
        searches=["single_pass"],
        qa_models=[reader],
        ks=[3, 5],
        percentile=90,
        run_dir=run_dir,
    )

    result["reader"] = reader_name
    quality_results.append(result)

    del result
    gc.collect()

quality = pd.concat(quality_results, ignore_index=True)
memory = pd.DataFrame(memory_results)

quality.to_csv(
    RESULTS_DIR / "focused_all_readers_quality.csv",
    index=False,
)

memory.to_csv(
    RESULTS_DIR / "focused_all_readers_memory.csv",
    index=False,
)

display(
    quality[
        [
            "dataset", "qa", "k", "questions",
            "exact_match", "f1",
            "mean_input_tokens",
            "qa_s_mean",
            "pipeline_stage_sum_s_mean",
        ]
    ]
)

display(memory)